# 2.4. Public JHS Congestion Analysis — Explainer

**Purpose:** This notebook analyzes seat congestion at public junior high schools (JHS) in NCR and Region IV-A, then optimizes student redistribution to minimize "aisle learners" (students without seats) using ESC policy levers.

**How to use this explainer:** Each section begins with context on *what* we're doing and *why*, followed by the original code. Look for **Discussion** blocks — these are prompts for team conversation about assumptions, limitations, and potential improvements.

---

### Notebook Roadmap

| Section | What It Does | Key Decision |
|---------|-------------|--------------|
| **1.0 Load Data** | Bring in 9 datasets (schools, distances, flow, seats, enrollment, coordinates, ESC slots, tuition, school size) | Which school year? Which regions? |
| **2.0 Build Congestion Table** | Filter to public JHS in NCR/IV-A, join enrollment + seats, impute missing seats, compute aisle learners | How to handle missing seat data? |
| **3.0 Prepare Optimization** | Build the menu of redirection options: which origin-private pairs are feasible? | What distance threshold? How to score options? |
| **4.0 Greedy Optimization** | Heuristic: address worst-congested schools first, pick nearest/cheapest private partner | Is greedy good enough? |
| **5.0 LP Optimization** | Global optimum via linear programming — same constraints, better allocation | How much better than greedy? |
| **6.0 Compare & Analyze** | Side-by-side comparison + blocked demand analysis for slot expansion priorities | Where should new ESC slots go? |
| **7.0 Export Payload** | Save all results for notebook 2.5 (report generation) | — |

---
# 0.0. Setup

Standard library imports. Nothing custom — all packages are widely available via pip/conda.

In [1]:
import os
import gc
import json
import copy
import requests
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

import networkx as nx
import geopandas as gpd
from shapely.geometry import Point
from difflib import SequenceMatcher

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from category_encoders import BinaryEncoder

from scipy.spatial import cKDTree
from scipy.sparse import lil_matrix
from scipy.sparse.csgraph import dijkstra
from pyproj import Transformer

from matplotlib.colors import Normalize                                                                                                                            
import matplotlib.cm as cm

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(PROJECT_DIR)

DATA_DIR = 'data'
DATA_PUBLIC_DIR = 'data/public'
DATA_PRIVATE_DIR = 'data/private'
DATA_PROCESSED_DIR = 'data/processed'
OUTPUT_DIR = 'output'

---
# 1.0. Load Data

We load **9 datasets** that together describe the supply side (schools, seats, ESC slots), the demand side (enrollment, student flows), and the spatial relationships (distances, coordinates, tuition costs).

**Key assumption:** All data is from **SY 2023-24**. Enrollment, flow, and seat data must align to the same year for the congestion calculation to be meaningful. If any dataset lags behind, our aisle learner counts will be off.

## 1.1. School Information

The master school registry — 61,442 schools nationwide. We slim it down to 8 essential columns because this notebook only needs identifiers, location, sector, and education level offerings.

**Discussion:** We use `old_region` (pre-2019 regional boundaries) because our seat and enrollment datasets still use the old classification. If DepEd migrates to the new regions, we'll need a crosswalk.

In [2]:
# Load school information                                                                                 
sch_info = pd.read_parquet('output/processed_project_bukas_school_information.parquet')                   
print(f"\nSchool info shape: {sch_info.shape}")                                                           
                                                                                                        
# Create slim version with just the columns we need   
essential_cols = [
    'school_id', 'school_name', 'old_region', 'division','sector',
    'offers_es','offers_jhs','offers_shs',
]
sch_info_slim = sch_info[essential_cols].copy()                   
sch_info_slim['school_id'] = sch_info_slim['school_id'].astype(str)                                       
                                                                                                        
print(f"School info slim shape: {sch_info_slim.shape}")                                                   
print(f"\nSample:")                                                                                       
display(sch_info_slim.head())


School info shape: (61442, 20)
School info slim shape: (61442, 8)

Sample:


,school_id,school_name,old_region,division,sector,offers_es,offers_jhs,offers_shs
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Public,True,False,False
1,100002,Bacarra CES,Region I,Ilocos Norte,Public,True,False,False
2,100003,Buyon ES,Region I,Ilocos Norte,Public,True,False,False
3,100004,Ganagan Elementary School,Region I,Ilocos Norte,Public,True,False,False
4,100005,Macupit ES,Region I,Ilocos Norte,Public,True,False,False


## 1.2. Distance Matrix

A pre-computed **8,331 × 8,331** road-network distance matrix (from notebook 2.2 via OSRM). This covers all schools in NCR and Region IV-A.

**Why road distances, not straight-line?** Families decide based on actual travel distance, not "as the crow flies." OSRM gives us realistic driving/commuting distances.

**Handling duplicates:** 24 integrated schools (offering both ES and JHS) have two entries in the matrix — one for each building's coordinates. We keep the first index and drop the second, then rebuild the index mapping.

**Discussion:**
- The 24 duplicates are a data artifact. Are there schools where the ES and JHS buildings are far enough apart that keeping both entries would matter?
- The distance matrix is symmetric (A→B = B→A). Should we consider asymmetric travel times (e.g., one-way streets, traffic patterns)?

In [3]:
# Load distance matrix
distance_matrix = np.load('output/school_distance_matrix_osrm.npy')
with open('output/school_distance_matrix_index.json', 'r') as f:
    index_data = json.load(f)
    school_ids = index_data['school_ids']
    school_id_to_idx = index_data['school_id_to_indices']
print(f"Distance matrix shape: {distance_matrix.shape}")

# Handle duplicate school IDs (keep first index, drop second)
# Some schools have 2 indices due to separate ES/JHS building coordinates
dupe_ids = [(schid, ids) for schid, ids in school_id_to_idx.items() if isinstance(ids, list) and len(ids) == 2]
idxs_to_drop = sorted([max(int(tup[1][0]), int(tup[1][1])) for tup in dupe_ids])
print(f"Duplicate IDs to resolve: {len(dupe_ids)}")

# Trim matrix by removing duplicate rows/columns
dm = distance_matrix.copy()
dm = np.delete(dm, idxs_to_drop, axis=1)
dm = np.delete(dm, idxs_to_drop, axis=0)
print(f"Trimmed distance matrix: {dm.shape}")

# Create old_idx -> new_idx mapping (accounting for dropped indices)
old_to_new_idx = {}
new_idx = 0
for old_idx in range(distance_matrix.shape[0]):
    if old_idx not in idxs_to_drop:
        old_to_new_idx[old_idx] = new_idx
        new_idx += 1

# Create school_id -> new_idx mapping
re_schids_to_idx = {}
for schid, idx_val in school_id_to_idx.items():
    # Get the first index (could be list or single value)
    if isinstance(idx_val, list):
        old_idx = int(idx_val[0])  # Keep first index
    else:
        old_idx = int(idx_val)
    
    # Map to new index if it wasn't dropped
    if old_idx in old_to_new_idx:
        re_schids_to_idx[schid] = old_to_new_idx[old_idx]

re_schids = list(re_schids_to_idx.keys())
print(f"Schools in distance matrix: {len(re_schids):,}")

del distance_matrix  # Free memory

Distance matrix shape: (13131, 13131)
Duplicate IDs to resolve: 25
Trimmed distance matrix: (13106, 13106)
Schools in distance matrix: 13,106


## 1.3. Student Flow

Grade 7 student flow data for SY 2023-24: which elementary school (origin) did each group of students come from, and which JHS (destination) did they enroll in?

Each row is an (origin, destination) pair with two counts:
- `count_non_beneficiary` — students NOT receiving ESC vouchers
- `count_esc_beneficiary` — students receiving ESC vouchers (already redirected to private schools)

**Why this matters:** Non-beneficiaries at congested public JHS are our **target population** — these are the students we could potentially redirect to private ESC schools.

In [4]:
student_flow = pd.read_parquet('output/grade_7_student_flow_table_sy2324.parquet')
print(f"Student flow shape: {student_flow.shape}")
print(f"Columns: {student_flow.columns.tolist()}")

Student flow shape: (288328, 4)
Columns: ['school_id_origin', 'school_id_destination', 'count_non_beneficiary', 'count_esc_beneficiary']


## 1.4. Public School Seats

Physical seat counts per school, broken down by education level. This is the **supply side** of our congestion equation.

**Limitation:** Seat data comes from DepEd's facilities inventory, which may not reflect recent construction or furniture delivery. Schools that received new seats after the survey date will appear more congested than they actually are.

In [5]:
public_seats = pd.read_parquet('output/processed_public_seats.parquet')
print(f"Public seats shape: {public_seats.shape}")

Public seats shape: (54360, 3)


## 1.5. JHS Enrollment

We filter the multi-year enrollment dataset to **SY 2023-24, junior high school only**, then aggregate by school. This gives us the **demand side** of the congestion equation.

**Why aggregate?** The raw data has rows per grade level (G7, G8, G9, G10). For congestion analysis, we care about total JHS enrollment per school — all four grade levels share the same physical seats.

In [6]:
enrollment = pd.read_parquet('output/processed_project_bukas_school_enrollment.parquet')

mask = (enrollment['school_year'] == '2023_24') & (enrollment['education_level'] == 'junior_high_school')
jhs_enrollment = enrollment[mask].groupby(['school_id'], as_index=False)['enrollment_count'].sum()
dict_jhs_enrollment = {schid: enr for schid, enr in zip(jhs_enrollment['school_id'], jhs_enrollment['enrollment_count'])}
print(f"JHS enrollment records: {len(dict_jhs_enrollment):,}")

JHS enrollment records: 16,376


## 1.6–1.8. Coordinates, ESC Slots, and Tuition

Three supporting datasets:

| Dataset | Purpose in This Notebook |
|---------|-------------------------|
| **Coordinates** | Not used directly in optimization, but loaded for potential spatial analysis |
| **ESC Slots** | How many voucher slots each private school has — this is the **capacity constraint** for redirection |
| **Tuition (GASTPE)** | JHS tuition fees at ESC-delivering schools — used to penalize expensive options in the priority score |

**Discussion:** Should we weight tuition more heavily? Lower-income families may be more sensitive to out-of-pocket costs even with ESC vouchers covering part of tuition.

In [7]:
coordinates = pd.read_parquet('output/all_schools_coordinates_ncr_region4a.parquet')
print(f"Coordinates shape: {coordinates.shape}")

Coordinates shape: (8331, 4)


In [8]:
esc_slots = pd.read_parquet('output/processed_esc_slots.parquet')
print(f"ESC slots shape: {esc_slots.shape}")

ESC slots shape: (4550, 7)


In [9]:
gastpe = pd.read_parquet('output/processed_gastpe_data.parquet')
print(f"GASTPE data shape: {gastpe.shape}")

# Filter to ESC-delivering schools with JHS tuition data
esc_tuition = gastpe[
    (gastpe['is_esc_delivering'] == True) & 
    (gastpe['average_jhs_total_fees'].notna())
][['school_id', 'average_jhs_total_fees']].copy()
esc_tuition.columns = ['school_id', 'tuition_jhs']
print(f"ESC schools with tuition data: {esc_tuition.shape[0]:,}")

GASTPE data shape: (5188, 5)
ESC schools with tuition data: 3,621


## 1.9. Private School Size (JHS Enrollment)

We calculate a **size bonus** for each private school based on its JHS enrollment, using **percentile rank** (0.0 = smallest, 1.0 = largest).

**Why prefer larger schools?**
- Larger schools are more likely to have the infrastructure (classrooms, teachers, facilities) to absorb additional students
- They tend to have more established academic programs
- Redirecting students to very small schools risks overwhelming their capacity

**Why percentile rank instead of raw enrollment?**
- Percentile rank is scale-invariant — it doesn't matter if enrollment numbers change across years
- It produces a bounded [0, 1] score that integrates cleanly into the priority formula
- No need for arbitrary binning (e.g., "small/medium/large") which loses granularity

**Discussion:** Is enrollment the best proxy for "absorptive capacity"? Alternatives worth considering:
- Number of classrooms or sections
- Teacher-to-student ratio
- Historical ESC absorption rate (how many beneficiaries they've successfully served before)

In [10]:
enrollment.head(1)

,school_id,enrollment_label,enrollment_count,grade_level,education_level,school_year
0,100001,enr_kindergarten,6.0,kindergarten,elementary,2022_23


In [11]:
# Filter to JHS enrollment for 2023_24 (all sectors)
mask_jhs = (
    (enrollment['school_year'] == '2023_24')
    & (enrollment['education_level'] == 'junior_high_school')
)
jhs_enr_all = enrollment[mask_jhs].groupby('school_id', as_index=False)['enrollment_count'].sum()

# Join with school info to get sector
jhs_enr_all = jhs_enr_all.merge(
    sch_info_slim[['school_id', 'sector']],
    on='school_id',
    how='left'
)

# Filter to private schools only
private_jhs_enrollment = jhs_enr_all[jhs_enr_all['sector'] == 'Private'].copy()
private_jhs_enrollment = private_jhs_enrollment[['school_id', 'enrollment_count']]
private_jhs_enrollment.columns = ['school_id', 'private_jhs_enrollment']
print(f"Private schools with JHS enrollment: {private_jhs_enrollment.shape[0]:,}")

# Calculate size_bonus using percentile rank (0.0 to 1.0)
# This gives each school a unique score based on where it falls in the distribution
private_jhs_enrollment['size_bonus'] = private_jhs_enrollment['private_jhs_enrollment'].rank(pct=True)

# Summary statistics
print(f"\nEnrollment distribution:")
print(f"  Min: {private_jhs_enrollment['private_jhs_enrollment'].min():,.0f} students")
print(f"  25th percentile: {private_jhs_enrollment['private_jhs_enrollment'].quantile(0.25):,.0f} students")
print(f"  Median: {private_jhs_enrollment['private_jhs_enrollment'].median():,.0f} students")
print(f"  75th percentile: {private_jhs_enrollment['private_jhs_enrollment'].quantile(0.75):,.0f} students")
print(f"  Max: {private_jhs_enrollment['private_jhs_enrollment'].max():,.0f} students")

print(f"\nSize bonus (percentile rank):")
print(f"  Range: [{private_jhs_enrollment['size_bonus'].min():.3f}, {private_jhs_enrollment['size_bonus'].max():.3f}]")
print(f"  Median school gets size_bonus ≈ 0.50")

# Show example mapping
print(f"\nExample size_bonus values:")
for pct in [0.1, 0.25, 0.5, 0.75, 0.9]:
    enr_at_pct = private_jhs_enrollment['private_jhs_enrollment'].quantile(pct)
    print(f"  {pct*100:.0f}th percentile ({enr_at_pct:,.0f} students) → size_bonus ≈ {pct:.2f}")

Private schools with JHS enrollment: 5,620

Enrollment distribution:
  Min: 0 students
  25th percentile: 39 students
  Median: 110 students
  75th percentile: 252 students
  Max: 2,490 students

Size bonus (percentile rank):
  Range: [0.018, 1.000]
  Median school gets size_bonus ≈ 0.50

Example size_bonus values:
  10th percentile (12 students) → size_bonus ≈ 0.10
  25th percentile (39 students) → size_bonus ≈ 0.25
  50th percentile (110 students) → size_bonus ≈ 0.50
  75th percentile (252 students) → size_bonus ≈ 0.75
  90th percentile (477 students) → size_bonus ≈ 0.90


---
# 2.0. Build Congestion Table

This is where we construct the core metric: **how many students lack seats at each public JHS?**

The formula is simple:
```
aisle_learners = max(0, enrollment − seats)
```

But getting there requires careful joins and imputation for missing data.

## 2.1. Filter Schools

We narrow from 61,442 schools nationwide to **~1,098 public JHS in NCR and Region IV-A**.

**Why these two regions?**
- NCR has the highest population density and most severe overcrowding
- Region IV-A (CALABARZON) is the most populous region outside NCR and experiences spillover congestion
- Together they represent the areas where ESC redirection would have the greatest impact

**Discussion:** Should we expand to other regions? Region III (Central Luzon) also has significant congestion. The framework generalizes — we'd just need distance matrices for those regions.

In [12]:
# Filter to public JHS in NCR and Region IV-A
mask = (
    (sch_info_slim['sector'] == 'Public')
    & (sch_info_slim['offers_jhs'] == True)
    & (sch_info_slim['old_region'].isin(['NCR', 'Region IV-A', 'Region III']))
)
sch_public_jhs = sch_info_slim.loc[mask].copy()
drop_cols = [col for col in sch_public_jhs.columns if re.search(r'offers_|sector', col)]
sch_public_jhs.drop(columns=drop_cols, inplace=True)
print(f"Public JHS in NCR/Region IV-A: {sch_public_jhs.shape[0]:,}")

Public JHS in NCR/Region IV-A: 2,032


## 2.2–2.3. Join Enrollment and Seat Data

We join JHS enrollment (demand) and seat counts (supply) to each school.

- **13 schools** are missing enrollment data — likely very new schools or data reporting gaps
- **64 schools** are missing seat data — these get imputed in the next step

**Assumption:** Schools missing enrollment are excluded from the congestion calculation entirely. If these schools are actually congested, we're undercounting aisle learners.

In [13]:
# Create enrollment lookup
dict_jhs_enrollment = jhs_enrollment.set_index('school_id')['enrollment_count'].to_dict()

# Join enrollment to school info
tmp_jn = sch_public_jhs.copy()
tmp_jn['enrollment_jhs'] = tmp_jn['school_id'].map(dict_jhs_enrollment)

# Check coverage
missing_enrollment = tmp_jn['enrollment_jhs'].isna().sum()
print(f"Schools with enrollment data: {tmp_jn.shape[0] - missing_enrollment:,}")
print(f"Schools missing enrollment: {missing_enrollment:,}")

Schools with enrollment data: 1,986
Schools missing enrollment: 46


In [14]:
# Filter seats to JHS level
jhs_seats = public_seats[public_seats['education_level'] == 'Junior High School'][['school_id', 'seat_count']].copy()

# Join seats
enr_seats = tmp_jn.merge(jhs_seats, on='school_id', how='left')

# Check coverage
missing_seats = enr_seats['seat_count'].isna().sum()
print(f"Schools with seat data: {enr_seats.shape[0] - missing_seats:,}")
print(f"Schools missing seats: {missing_seats:,}")

Schools with seat data: 1,870
Schools missing seats: 162


## 2.4. Impute Missing Seats (KNN)

64 schools are missing seat data. Rather than dropping them (which would undercount congestion), we **impute** using K-Nearest Neighbors.

**How KNN imputation works here:**
1. Encode region and division as numeric features
2. Use `[region, division, enrollment]` as the feature space
3. For each school with missing seats, find the 5 nearest schools (by these features) that DO have seat data
4. Average their seat counts as the imputed value

**Why KNN over simpler methods?**
- Mean/median imputation ignores local context (a small rural school vs. a large urban school have very different seat counts)
- KNN respects the structure: schools in the same division with similar enrollment likely have similar seat counts

**Discussion:**
- Is k=5 the right number of neighbors? Too few → noisy, too many → over-smoothed
- Should we weight neighbors by distance (closer neighbors count more)?
- Alternative: use classroom count × seats-per-classroom as a proxy?

In [15]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

df_imp = enr_seats.copy()

# Identify rows needing imputation
mask = df_imp['seat_count'].isna()
print(f"Rows to impute: {mask.sum():,}")

# Prepare features for KNN
cat_cols = ['old_region', 'division']
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_imp[f'{col}_enc'] = le.fit_transform(df_imp[col].astype(str))
    le_dict[col] = le

# KNN imputation
feature_cols = ['old_region_enc', 'division_enc', 'enrollment_jhs']
imputer = KNNImputer(n_neighbors=5)

# Fit on rows with seat data, transform all
df_for_impute = df_imp[feature_cols + ['seat_count']].copy()
df_imputed_arr = imputer.fit_transform(df_for_impute)
df_imp['seat_count'] = df_imputed_arr[:, -1]

# Track imputed rows
df_imp['is_seats_imputed'] = mask

# Clean up
df_imp = df_imp.drop(columns=[f'{col}_enc' for col in cat_cols])
print(f"Imputation complete. Imputed: {df_imp['is_seats_imputed'].sum():,}")

Rows to impute: 162
Imputation complete. Imputed: 162


## 2.5. Calculate Congestion Metrics

Two metrics per school:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| `seat_utilization_jhs` | enrollment / seats | >1.0 = overcrowded |
| `count_aisle_learner_jhs` | max(0, enrollment − seats) | Students without a seat |

**Key result:** ~577,700 total aisle learners across ~715 congested schools. This is the **baseline** we're trying to reduce.

**Limitation:** "Aisle learner" is a simplification. In reality, schools use shift systems (morning/afternoon classes) to manage overcrowding. Our metric assumes all students need seats simultaneously, which overstates the problem but reflects the infrastructure gap.

In [16]:
df_cong = df_imp.copy()

# Calculate congestion metrics
df_cong['seat_utilization_jhs'] = df_cong['enrollment_jhs'] / df_cong['seat_count']
df_cong['count_aisle_learner_jhs'] = (df_cong['enrollment_jhs'] - df_cong['seat_count']).clip(lower=0)

# Summary statistics
total_schools = df_cong.shape[0]
congested = (df_cong['count_aisle_learner_jhs'] > 0).sum()
total_aisle = df_cong['count_aisle_learner_jhs'].sum()

print(f"Total public JHS analyzed: {total_schools:,}")
print(f"Congested schools (aisle learners > 0): {congested:,} ({congested/total_schools*100:.1f}%)")
print(f"Total aisle learners: {total_aisle:,.0f}")

Total public JHS analyzed: 2,032
Congested schools (aisle learners > 0): 1,267 (62.4%)
Total aisle learners: 768,463


---
# 3.0. Prepare Optimization Data

Now we build the **menu of redirection options**. The question: for each student flow going to a congested public JHS, which nearby private ESC schools could absorb some of those students?

This section constructs the feasibility set by applying three filters:
1. **Geographic**: Private school must be within 5km of the origin ES (road distance)
2. **Capacity**: Private school must have available ESC slots
3. **Flow-based**: Students must actually exist in that origin→destination flow

## 3.1–3.2. Filter ESC Schools and Student Flow

We restrict both ESC schools and student flows to schools present in our distance matrix (i.e., schools in NCR, Region IV-A, and Region III).

In [17]:
# Filter ESC schools to NCR, Region IV-A, and Region III (schools in distance matrix)
mask = esc_slots['school_id'].isin(re_schids)
esc_in_loc = esc_slots[['school_id', 'total_count_slots']].loc[mask].copy()
print(f"ESC schools in NCR, Region IV-A, & Region III: {esc_in_loc.shape[0]:,}")

ESC schools in NCR, Region IV-A, & Region III: 1,402


In [18]:
# Filter flow to schools in NCR/Region IV-A
mask = (
    (student_flow['school_id_origin'].isin(re_schids))
    & (student_flow['school_id_destination'].isin(re_schids))
)
flow = student_flow[mask].copy()
print(f"Flow pairs in location: {flow.shape[0]:,}")

Flow pairs in location: 84,131


In [24]:
# flow['school_id_origin'].nunique(), flow['school_id_destination'].nunique()

(9941, 4234)

## 3.3. Build Distance Lookup (Origin ES → Private ESC)

For each origin elementary school, we find all private ESC schools within **5 km** road distance.

**Why 5 km?** This is a policy-relevant threshold — DepEd's ESC program targets nearby private schools as alternatives. Beyond 5 km, the commute burden on families becomes unreasonable, especially for JHS-age students.

**Important:** The distance here is from the **origin ES** to the private school, NOT from the congested public JHS to the private school. This is because students are being redirected *before* they enroll in JHS — they're choosing where to go for Grade 7 while still at their elementary school.

**Discussion:**
- Is 5 km the right threshold? Urban NCR families might tolerate shorter distances; suburban CALABARZON families might accept longer ones.
- Should the threshold be travel-time-based (e.g., 30 minutes by public transport) rather than distance-based?
- What percentage of potential redirections are we losing by using 5 km vs. 10 km?

In [19]:
# Get private ESC school IDs in distance matrix
private_esc_ids = esc_in_loc['school_id'].tolist()
private_in_dm = [sid for sid in private_esc_ids if sid in re_schids_to_idx]

# Get origin school IDs from flow data
public_jhs_ids = df_cong['school_id'].tolist()
origin_ids = flow[flow['school_id_destination'].isin(public_jhs_ids)]['school_id_origin'].unique().tolist()
origins_in_dm = [sid for sid in origin_ids if sid in re_schids_to_idx]

print(f"Private ESC schools in distance matrix: {len(private_in_dm):,}")
print(f"Origin schools in distance matrix: {len(origins_in_dm):,}")

# Build origin-to-private distance lookup
MAX_DISTANCE_KM = 5  # Only consider private schools within 5km of origin

rows = []
for origin_id in origins_in_dm:
    origin_idx = re_schids_to_idx[origin_id]
    for private_id in private_in_dm:
        private_idx = re_schids_to_idx[private_id]
        distance_m = dm[origin_idx, private_idx]
        distance_km = distance_m / 1000
        
        if distance_km <= MAX_DISTANCE_KM and distance_m < np.inf:
            rows.append({
                'school_id_origin': origin_id,
                'private_esc_id': private_id,
                'distance_to_private_m': distance_m,
                'distance_to_private_km': distance_km
            })

origin_to_private = pd.DataFrame(rows)
origin_to_private = origin_to_private.merge(
    esc_in_loc[['school_id', 'total_count_slots']],
    left_on='private_esc_id',
    right_on='school_id',
    how='left'
).drop(columns='school_id')

print(f"Origin-private pairs within {MAX_DISTANCE_KM}km: {origin_to_private.shape[0]:,}")

Private ESC schools in distance matrix: 1,402
Origin schools in distance matrix: 8,613
Origin-private pairs within 5km: 65,481


## 3.4–3.5. Link Flows to Congested Schools and Build Redirection Options

We connect two pieces:
1. Which student flows go to **congested** public JHS? (flows where destination has aisle learners > 0)
2. Which **private ESC schools** are near the origin of those flows?

The merge produces the full set of redirection options: "These X non-beneficiary students from origin A, currently heading to congested JHS B, could instead go to nearby private ESC C."

In [20]:
# Join flow data with congestion info
flow_to_congested = flow.merge(
    df_cong[['school_id', 'count_aisle_learner_jhs', 'seat_utilization_jhs', 'enrollment_jhs', 'seat_count']],
    left_on='school_id_destination',
    right_on='school_id',
    how='inner'
).drop(columns='school_id')

# Filter to only congested destinations
mask = flow_to_congested['count_aisle_learner_jhs'] > 0
flow_to_congested = flow_to_congested[mask].copy()

print(f"Flows to congested public JHS: {flow_to_congested.shape[0]:,}")
print(f"Total non-beneficiaries in these flows: {flow_to_congested['count_non_beneficiary'].sum():,.0f}")

Flows to congested public JHS: 36,164
Total non-beneficiaries in these flows: 400,936


In [21]:
# Create redirection options: (origin, destination, private_esc) combinations
redirection_options = flow_to_congested.merge(
    origin_to_private,
    on='school_id_origin',
    how='inner'
)
print(f"Redirection options: {redirection_options.shape[0]:,}")

Redirection options: 405,414


## 3.6. Calculate Available ESC Slots

A private school's **available slots** = total ESC slots allocated − students already using them (current beneficiaries).

If a school has 100 slots and 80 current beneficiaries, only 20 slots are available for new redirections.

**Discussion:** The "current beneficiaries" count comes from the student flow data. Are there timing mismatches? If slots were allocated for SY 2024-25 but beneficiaries are counted from SY 2023-24, our availability estimates could be off.

In [22]:
# Calculate current beneficiaries per ESC school
current_benef = flow.groupby('school_id_destination', as_index=False)['count_esc_beneficiary'].sum()
current_benef.columns = ['school_id', 'current_beneficiaries']
current_benef['current_beneficiaries'] = current_benef['current_beneficiaries'].fillna(0)

# Calculate available slots
esc_available = esc_in_loc.merge(current_benef, on='school_id', how='left')
esc_available['current_beneficiaries'] = esc_available['current_beneficiaries'].fillna(0)
esc_available['available_slots'] = esc_available['total_count_slots'] - esc_available['current_beneficiaries']
esc_available.loc[esc_available['available_slots'] < 0, 'available_slots'] = 0

print(f"ESC schools with available slots > 0: {(esc_available['available_slots'] > 0).sum():,}")
print(f"Total available slots: {esc_available['available_slots'].sum():,.0f}")

# Update redirection options with available slots
redirection_options = redirection_options.drop(columns=['total_count_slots'], errors='ignore')
redirection_options = redirection_options.merge(
    esc_available[['school_id', 'available_slots']],
    left_on='private_esc_id',
    right_on='school_id',
    how='left'
).drop(columns='school_id')

# Filter to options with available slots
redirection_options = redirection_options[redirection_options['available_slots'] > 0].copy()
print(f"Redirection options with available slots: {redirection_options.shape[0]:,}")

ESC schools with available slots > 0: 1,266
Total available slots: 37,489
Redirection options with available slots: 355,716


## 3.7. Priority Scoring

Each redirection option gets a **priority score** that combines three factors:

```
priority_score = norm_distance + 0.5 × norm_tuition − 0.3 × size_bonus
```

| Factor | Weight | Direction | Rationale |
|--------|--------|-----------|-----------|
| Distance | 1.0 | Higher = worse | Closer schools are more convenient for families |
| Tuition | 0.5 | Higher = worse | Cheaper schools reduce financial burden (even with ESC voucher) |
| School size | 0.3 | Higher = better (subtracted) | Larger schools can better absorb additional students |

**Lower score = better option** (nearer, cheaper, larger).

Normalization is done **per destination group** — each congested public JHS gets its own min-max scaling. This ensures the priority ranking is relative within each school's set of options, not globally biased toward areas with more private schools.

**Discussion:**
- The weights (1.0, 0.5, 0.3) are manually chosen. Should we calibrate them empirically (e.g., from revealed preference data on where families actually choose to go)?
- Should we add a "quality" dimension (e.g., school performance indicators)?
- The normalization per destination group means a "good" option near one school may look different from a "good" option near another. Is this desirable?

In [23]:
# Join tuition data
redirection_options = redirection_options.merge(
    esc_tuition,
    left_on='private_esc_id',
    right_on='school_id',
    how='left'
).drop(columns='school_id', errors='ignore')

# Impute missing tuition with median
median_tuition = redirection_options['tuition_jhs'].median()
redirection_options['tuition_jhs'] = redirection_options['tuition_jhs'].fillna(median_tuition)
print(f"Median JHS tuition: ₱{median_tuition:,.0f}")

# Join school size data (size_bonus from percentile rank)
redirection_options = redirection_options.merge(
    private_jhs_enrollment[['school_id', 'private_jhs_enrollment', 'size_bonus']],
    left_on='private_esc_id',
    right_on='school_id',
    how='left'
).drop(columns='school_id', errors='ignore')

# Handle missing enrollment (schools not in enrollment data)
redirection_options['private_jhs_enrollment'] = redirection_options['private_jhs_enrollment'].fillna(0)
redirection_options['size_bonus'] = redirection_options['size_bonus'].fillna(0.0)  # No bonus for missing

# Summary of size_bonus distribution
print(f"\nSize bonus distribution in redirection options:")
print(f"  Schools with size data: {redirection_options['size_bonus'].gt(0).sum():,}")
print(f"  Schools missing size data: {redirection_options['size_bonus'].eq(0).sum():,}")
print(f"  Mean size_bonus: {redirection_options['size_bonus'].mean():.3f}")
print(f"  Median size_bonus: {redirection_options['size_bonus'].median():.3f}")

# Priority scoring parameters
TUITION_WEIGHT = 0.5   # Weight for tuition in priority score
SIZE_WEIGHT = 0.3      # Weight for size bonus (multiplied by size_bonus 0-1)

# Calculate normalized values per destination group
for dest_id in redirection_options['school_id_destination'].unique():
    mask = redirection_options['school_id_destination'] == dest_id
    options = redirection_options.loc[mask]
    
    if len(options) > 1:
        dist_min = options['distance_to_private_km'].min()
        dist_max = options['distance_to_private_km'].max()
        tuition_min = options['tuition_jhs'].min()
        tuition_max = options['tuition_jhs'].max()
        
        redirection_options.loc[mask, 'norm_distance'] = (
            (options['distance_to_private_km'] - dist_min) / (dist_max - dist_min + 1e-6)
        )
        redirection_options.loc[mask, 'norm_tuition'] = (
            (options['tuition_jhs'] - tuition_min) / (tuition_max - tuition_min + 1e-6)
        )
    else:
        redirection_options.loc[mask, 'norm_distance'] = 0
        redirection_options.loc[mask, 'norm_tuition'] = 0

# Calculate priority score (lower = better)
# Base: distance + tuition penalty
# Bonus: subtract SIZE_WEIGHT * size_bonus (larger schools get bigger subtraction)
redirection_options['priority_score'] = (
    redirection_options['norm_distance'] + 
    TUITION_WEIGHT * redirection_options['norm_tuition'] -
    SIZE_WEIGHT * redirection_options['size_bonus']
)

print(f"\nPriority score calculated for {redirection_options.shape[0]:,} options")
print(f"Priority score range: [{redirection_options['priority_score'].min():.3f}, {redirection_options['priority_score'].max():.3f}]")
print(f"  Formula: norm_distance + {TUITION_WEIGHT}×norm_tuition - {SIZE_WEIGHT}×size_bonus")
print(f"  size_bonus = percentile rank [0.0, 1.0]")

Median JHS tuition: ₱39,805

Size bonus distribution in redirection options:
  Schools with size data: 284,670
  Schools missing size data: 0
  Mean size_bonus: 0.642
  Median size_bonus: 0.662

Priority score calculated for 284,670 options
Priority score range: [-0.299, 1.436]
  Formula: norm_distance + 0.5×norm_tuition - 0.3×size_bonus
  size_bonus = percentile rank [0.0, 1.0]


---
# 4.0. Greedy Optimization

The greedy algorithm is a **heuristic** — it makes locally optimal choices without guaranteeing a globally optimal solution.

**Algorithm:**
1. Sort all redirection options: most congested destination first, then lowest priority score (nearest/cheapest/largest)
2. Walk through the sorted list. For each option:
   - Check: does the private school still have slots? Does the public JHS still have aisle learners? Are there non-beneficiaries left in this flow?
   - If yes to all three, redirect as many students as possible (minimum of the three capacities)
   - Update the remaining capacities

**Why start with greedy?** It's fast, interpretable, and provides a baseline. The LP optimization (Section 5) will show how much improvement is possible with global optimization.

**Key constraint — flow tracking:** We track `remaining_non_benef` per (origin, destination) pair. This prevents the algorithm from redirecting more students from a flow than actually exist. Without this, the same students could be "double-counted" across multiple private school options.

In [24]:
# Initialize remaining capacity tracking
remaining_slots = esc_available.set_index('school_id')['available_slots'].to_dict()
remaining_aisle = df_cong.set_index('school_id')['count_aisle_learner_jhs'].to_dict()

# Track remaining non-beneficiaries per (origin, destination) flow
# This prevents over-redirecting from a single flow
remaining_non_benef = flow_to_congested.set_index(
    ['school_id_origin', 'school_id_destination']
)['count_non_beneficiary'].to_dict()

# Sort redirection options by: most congested destination first, then lowest priority score
redirection_options_sorted = redirection_options.sort_values(
    by=['count_aisle_learner_jhs', 'priority_score'],
    ascending=[False, True]
)

print(f"Initialized tracking for {len(remaining_slots):,} private schools")
print(f"Initialized tracking for {len(remaining_aisle):,} congested public schools")
print(f"Initialized tracking for {len(remaining_non_benef):,} flows")

Initialized tracking for 1,058 private schools
Initialized tracking for 1,098 congested public schools
Initialized tracking for 22,879 flows


In [25]:
redirection_log = []

for _, row in redirection_options_sorted.iterrows():
    origin_id = row['school_id_origin']
    pub_jhs_id = row['school_id_destination']
    private_id = row['private_esc_id']
    distance_km = row['distance_to_private_km']
    tuition = row['tuition_jhs']
    priority = row['priority_score']
    
    # Check remaining capacity
    slots_left = remaining_slots.get(private_id, 0)
    aisle_left = remaining_aisle.get(pub_jhs_id, 0)
    
    # Get remaining non-beneficiaries for THIS specific flow
    flow_key = (origin_id, pub_jhs_id)
    non_benef_left = remaining_non_benef.get(flow_key, 0)
    
    if slots_left <= 0 or aisle_left <= 0 or non_benef_left <= 0:
        continue
    
    # Calculate students to redirect (use remaining non-beneficiaries, not original count)
    count_to_redirect = min(slots_left, aisle_left, non_benef_left)
    
    if count_to_redirect > 0:
        # Log the redirection
        redirection_log.append({
            'school_id_origin': origin_id,
            'school_id_destination': pub_jhs_id,
            'private_esc_id': private_id,
            'count_redirected': count_to_redirect,
            'distance_to_private_km': distance_km,
            'tuition_jhs': tuition,
            'priority_score': priority
        })
        
        # Update tracking
        remaining_slots[private_id] -= count_to_redirect
        remaining_aisle[pub_jhs_id] -= count_to_redirect
        remaining_non_benef[flow_key] -= count_to_redirect

redirections = pd.DataFrame(redirection_log)
print(f"Total redirections logged: {len(redirections):,}")
print(f"Total students redirected: {redirections['count_redirected'].sum():,.0f}")

Total redirections logged: 5,070
Total students redirected: 25,807


In [26]:
# Update congestion table with Greedy results
scenario1_congestion = df_cong.copy()
greedy_by_dest = redirections.groupby('school_id_destination')['count_redirected'].sum().to_dict()

scenario1_congestion['redirected_greedy'] = scenario1_congestion['school_id'].map(greedy_by_dest).fillna(0)
scenario1_congestion['aisle_learners_greedy'] = (
    scenario1_congestion['count_aisle_learner_jhs'] - scenario1_congestion['redirected_greedy']
).clip(lower=0)

# Summary
baseline_aisle = df_cong['count_aisle_learner_jhs'].sum()
greedy_aisle = scenario1_congestion['aisle_learners_greedy'].sum()

print(f"\nGREEDY OPTIMIZATION RESULTS:")
print(f"  Baseline aisle learners: {baseline_aisle:,.0f}")
print(f"  After Greedy: {greedy_aisle:,.0f}")
print(f"  Reduction: {baseline_aisle - greedy_aisle:,.0f} ({((baseline_aisle - greedy_aisle)/baseline_aisle)*100:.1f}%)")
print(f"  Private ESC schools used: {redirections['private_esc_id'].nunique():,}")


GREEDY OPTIMIZATION RESULTS:
  Baseline aisle learners: 577,701
  After Greedy: 551,894
  Reduction: 25,807 (4.5%)
  Private ESC schools used: 947


---
# 5.0. LP Optimization

The LP (Linear Programming) optimization finds the **globally optimal** redistribution — the allocation that maximizes total weighted redirections while respecting all constraints simultaneously.

**How LP differs from Greedy:**
- Greedy processes options one at a time in a fixed order → gets stuck in local optima
- LP considers ALL options simultaneously → finds the global optimum
- Both respect the same three constraints (slots, aisle learners, flow limits)

**Why "weighted" redirections?** The objective isn't just "maximize students redirected" — it also penalizes distant and expensive options while rewarding larger schools. This ensures the LP doesn't fill up a far-away expensive school just because it has lots of slots.

**LP is independent of Greedy.** Both start from the same baseline data (`esc_available`, `df_cong`, `flow_to_congested`). The LP does NOT build on Greedy's results — it solves the same problem from scratch with a different algorithm.

In [27]:
# Install PuLP if needed
try:
    from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus, value
except ImportError:
    !pip install -qq pulp
    from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus, value

# Prepare constraint dictionaries
flow_limits = flow_to_congested.groupby(
    ['school_id_origin', 'school_id_destination']
)['count_non_beneficiary'].first().to_dict()

slot_limits = esc_available.set_index('school_id')['available_slots'].to_dict()
aisle_limits = df_cong.set_index('school_id')['count_aisle_learner_jhs'].to_dict()

print(f"Flow constraints: {len(flow_limits):,}")
print(f"Slot constraints: {len(slot_limits):,}")
print(f"Aisle learner constraints: {len(aisle_limits):,}")

Flow constraints: 22,879
Slot constraints: 1,058
Aisle learner constraints: 1,098


## 5.2–5.3. Decision Variables and Constraints

**Decision variables:** One variable `x[i]` per redirection option — "how many students to redirect along this option." Each variable is bounded between 0 and the minimum of (non-beneficiaries, available slots, aisle learners).

**Three constraint families:**

| Constraint | What It Prevents | Count |
|-----------|-----------------|-------|
| **Flow** | Redirecting more students from a flow than exist | ~22,879 |
| **Slot** | Exceeding a private school's available ESC slots | ~1,058 |
| **Aisle** | Redirecting more students than a public JHS has as aisle learners | ~1,098 |

Total: ~21,000 constraints over ~284,670 variables. This is a medium-sized LP that solves in about 1-2 seconds.

In [28]:
# Create LP problem
prob = LpProblem("Maximize_Student_Redistribution", LpMaximize)

# Create decision variables: x[i] = students redirected for option i
x = {}
for idx, row in redirection_options.iterrows():
    var_name = f"x_{idx}"
    max_val = min(
        row['count_non_beneficiary'],
        row['available_slots'],
        row['count_aisle_learner_jhs']
    )
    x[idx] = LpVariable(var_name, lowBound=0, upBound=max_val, cat='Continuous')

print(f"Decision variables created: {len(x):,}")

Decision variables created: 284,670


In [29]:
# Constraint 1: Flow limit (can't redirect more than non-beneficiaries per flow)
flow_groups = redirection_options.groupby(['school_id_origin', 'school_id_destination'])
for (origin, dest), group in flow_groups:
    flow_key = (origin, dest)
    if flow_key in flow_limits:
        prob += lpSum([x[idx] for idx in group.index]) <= flow_limits[flow_key], f"flow_{origin}_{dest}"

# Constraint 2: Slot limit (can't exceed available slots per private school)
private_groups = redirection_options.groupby('private_esc_id')
for private_id, group in private_groups:
    if private_id in slot_limits:
        prob += lpSum([x[idx] for idx in group.index]) <= slot_limits[private_id], f"slots_{private_id}"

# Constraint 3: Aisle learner limit (can't redirect more than aisle learners per public school)
dest_groups = redirection_options.groupby('school_id_destination')
for dest_id, group in dest_groups:
    if dest_id in aisle_limits:
        prob += lpSum([x[idx] for idx in group.index]) <= aisle_limits[dest_id], f"aisle_{dest_id}"

print(f"Constraints added: {len(prob.constraints):,}")

Constraints added: 21,011


## 5.4. Objective Function

The LP maximizes:

```
Σ coef[i] × x[i]
```

Where each coefficient balances four factors:

```
coef = 1.0 − 0.3 × norm_distance − 0.3 × norm_tuition + 0.3 × size_bonus
```

- **Base value: 1.0** per student — every redirection is worth something
- **Distance penalty: −0.3** — prefer closer schools
- **Cost penalty: −0.3** — prefer cheaper schools
- **Size bonus: +0.3 × percentile rank** — prefer larger schools (a school at the 90th percentile gets +0.27, median gets +0.15)

The coefficient range is roughly [0.1, 1.3], so the LP will always prefer to redirect *someone* over nobody, but will choose closer/cheaper/larger options when possible.

In [30]:
COST_PENALTY = 0.3      # Penalty for expensive schools
DISTANCE_PENALTY = 0.3  # Penalty for distant schools
SIZE_WEIGHT_LP = 0.3    # Weight for size bonus (multiplied by size_bonus 0-1)

# Objective: Maximize redirections with distance/cost penalties and size bonus
objective_terms = []
for idx, row in redirection_options.iterrows():
    # Base value: 1 per student redirected
    # Penalties: deductions for distance and cost
    # Bonus: addition proportional to size_bonus (0-1)
    coef = (
        1.0 
        - (DISTANCE_PENALTY * row['norm_distance']) 
        - (COST_PENALTY * row['norm_tuition'])
        + (SIZE_WEIGHT_LP * row['size_bonus'])
    )
    objective_terms.append(coef * x[idx])

prob += lpSum(objective_terms), "Total_Weighted_Redirections"
print(f"Objective function defined with {len(objective_terms):,} terms")
print(f"  - Distance penalty: {DISTANCE_PENALTY}")
print(f"  - Cost penalty: {COST_PENALTY}")
print(f"  - Size weight: {SIZE_WEIGHT_LP} × size_bonus")
print(f"    (Decile 10 gets +{SIZE_WEIGHT_LP:.2f}, Decile 5 gets +{SIZE_WEIGHT_LP * 0.44:.2f}, Decile 1 gets +0.00)")

Objective function defined with 284,670 terms
  - Distance penalty: 0.3
  - Cost penalty: 0.3
  - Size weight: 0.3 × size_bonus
    (Decile 10 gets +0.30, Decile 5 gets +0.13, Decile 1 gets +0.00)


In [31]:
print("Solving LP...")
prob.solve()

print(f"Status: {LpStatus[prob.status]}")
print(f"Objective value: {value(prob.objective):,.2f}")

Solving LP...
Welcome to the CBC MILP Solver 
Version: 2.10.10 
Build Date: Sep 26 2023 

command line - /home/jupyter/.local/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/arm64/cbc /tmp/f8f28d8905854747bcfe2c408c77628e-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/f8f28d8905854747bcfe2c408c77628e-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 21016 COLUMNS
At line 1159697 RHS
At line 1180709 BOUNDS
At line 1465380 ENDATA
Problem MODEL has 21011 rows, 284670 columns and 854010 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 19106 (-1905) rows, 283406 (-1264) columns and 793278 (-60732) elements
0  Obj 228.05813 Dual inf 261937.23 (283405)
320  Obj 725611.11 Primal inf 1732366.9 (18565)
640  Obj 587831.47 Primal inf 1302211.1 (18116)
960  Obj 481813.28 Primal inf 1006156.5 (17481)
1280  Obj 418749.42 Primal inf 837303.88 (17015)
1600  Obj 367199.04 Prima

In [32]:
# Extract LP solution
redirection_options_lp = redirection_options.copy()
redirection_options_lp['count_redirected_lp'] = [value(x[idx]) for idx in redirection_options.index]

# Filter to positive redirections
redirections_lp = redirection_options_lp[redirection_options_lp['count_redirected_lp'] > 0].copy()

print(f"Total students redirected (LP): {redirections_lp['count_redirected_lp'].sum():,.0f}")

# Update congestion table with LP results
scenario1_lp_congestion = df_cong.copy()
redirections_by_dest_lp = redirections_lp.groupby('school_id_destination')['count_redirected_lp'].sum().to_dict()

scenario1_lp_congestion['redirected_lp'] = scenario1_lp_congestion['school_id'].map(redirections_by_dest_lp).fillna(0)
scenario1_lp_congestion['aisle_learners_lp'] = (
    scenario1_lp_congestion['count_aisle_learner_jhs'] - scenario1_lp_congestion['redirected_lp']
).clip(lower=0)

# Summary
lp_aisle = scenario1_lp_congestion['aisle_learners_lp'].sum()

print(f"\nLP OPTIMIZATION RESULTS:")
print(f"  Baseline aisle learners: {baseline_aisle:,.0f}")
print(f"  After LP: {lp_aisle:,.0f}")
print(f"  Reduction: {baseline_aisle - lp_aisle:,.0f} ({((baseline_aisle - lp_aisle)/baseline_aisle)*100:.1f}%)")
print(f"  Private ESC schools used: {redirections_lp['private_esc_id'].nunique():,}")

Total students redirected (LP): 25,930

LP OPTIMIZATION RESULTS:
  Baseline aisle learners: 577,701
  After LP: 551,771
  Reduction: 25,930 (4.5%)
  Private ESC schools used: 947


---
# 6.0. Compare Results

Side-by-side comparison of Greedy vs LP. Key things to notice:

- **Students redirected:** LP redirects slightly more (~25,930 vs ~25,807) — the global optimization finds a few hundred more feasible redirections
- **Average distance:** LP achieves **much shorter** average distances (~1.0 km vs ~2.5 km) — this is LP's main advantage. By optimizing globally, it can match students to closer schools instead of "using up" nearby slots on the first congested school it encounters
- **Average tuition:** Nearly identical — both algorithms pick similarly priced schools

**The takeaway:** LP doesn't dramatically increase the *number* of redirections (both are constrained by the same ~27K available slots), but it produces a **higher-quality** allocation with shorter commutes.

**Discussion:**
- The ~4.5% reduction seems small relative to ~577K aisle learners. What are the binding constraints? (Spoiler: it's mostly ESC slot availability — see Section 6.1)
- Should we present Greedy or LP results to stakeholders? LP is better but harder to explain. Greedy is intuitive ("we help the most crowded schools first").

In [33]:
print("="*70)
print("COMPARISON: GREEDY vs LP OPTIMIZATION")
print("="*70)
print(f"{'Metric':<35} {'Greedy':>15} {'LP':>15}")
print("-"*70)
print(f"{'Students redirected':<35} {redirections['count_redirected'].sum():>15,.0f} {redirections_lp['count_redirected_lp'].sum():>15,.0f}")
print(f"{'Aisle learners remaining':<35} {greedy_aisle:>15,.0f} {lp_aisle:>15,.0f}")
print(f"{'Reduction from baseline':<35} {(baseline_aisle-greedy_aisle)/baseline_aisle*100:>14.3f}% {(baseline_aisle-lp_aisle)/baseline_aisle*100:>14.3f}%")
print(f"{'Private ESC schools used':<35} {redirections['private_esc_id'].nunique():>15,} {redirections_lp['private_esc_id'].nunique():>15,}")

# Average distance comparison
greedy_avg_dist = np.average(redirections['distance_to_private_km'], weights=redirections['count_redirected'])
lp_avg_dist = np.average(redirections_lp['distance_to_private_km'], weights=redirections_lp['count_redirected_lp'])
print(f"{'Avg redirect distance (km)':<35} {greedy_avg_dist:>15.2f} {lp_avg_dist:>15.2f}")

# Average tuition comparison
greedy_avg_tuition = np.average(redirections['tuition_jhs'], weights=redirections['count_redirected'])
lp_avg_tuition = np.average(redirections_lp['tuition_jhs'], weights=redirections_lp['count_redirected_lp'])
print(f"{'Avg tuition (PHP)':<35} {greedy_avg_tuition:>15,.0f} {lp_avg_tuition:>15,.0f}")
print("="*70)

COMPARISON: GREEDY vs LP OPTIMIZATION
Metric                                       Greedy              LP
----------------------------------------------------------------------
Students redirected                          25,807          25,930
Aisle learners remaining                    551,894         551,771
Reduction from baseline                      4.467%          4.488%
Private ESC schools used                        947             947
Avg redirect distance (km)                     2.51            1.03
Avg tuition (PHP)                            38,633          38,795


# 6.1. Blocked Demand Analysis

After LP optimization, ~551,800 aisle learners remain. Why couldn't we redirect more?

The **binding constraint** is ESC slot availability. This section identifies private schools that:
1. Used ≥95% of their available slots (capacity-constrained)
2. Could serve more students if they received additional ESC slot allocations

**`unique_blocked_demand`** counts students who are *exclusively* assigned to each constrained school (using the priority score to assign each flow to its best option). This avoids double-counting students across schools.

**Policy implication:** The top schools in this list are the highest-priority candidates for ESC slot expansion — they're in the right locations, at reasonable prices, and have demonstrated demand.

In [34]:
# Step 1: Calculate slots used by LP for each private ESC
slots_used_by_school = redirections_lp.groupby('private_esc_id')['count_redirected_lp'].sum()

# Step 2: Build capacity info for private ESC schools
capacity_info = esc_available[['school_id', 'total_count_slots', 'available_slots']].copy()
capacity_info.columns = ['private_esc_id', 'total_slots', 'available_slots_before']
capacity_info['slots_used_lp'] = capacity_info['private_esc_id'].map(slots_used_by_school).fillna(0)
capacity_info['utilization_rate'] = capacity_info['slots_used_lp'] / capacity_info['available_slots_before']
capacity_info['utilization_rate'] = capacity_info['utilization_rate'].replace([float('inf')], 0)

# Step 3: Identify capacity-constrained schools (>= 95% utilized)
constrained_schools = set(capacity_info[capacity_info['utilization_rate'] >= 0.95]['private_esc_id'].tolist())
print(f"Capacity-constrained private ESC schools (>=95% utilized): {len(constrained_schools):,}")

Capacity-constrained private ESC schools (>=95% utilized): 931


In [35]:
# Step 4: Calculate remaining demand after LP optimization
remaining_aisle_lp = scenario1_lp_congestion.set_index('school_id')['aisle_learners_lp'].to_dict()

lp_redirections_by_flow = redirections_lp.groupby(
    ['school_id_origin', 'school_id_destination']
)['count_redirected_lp'].sum()

flow_remaining = flow_to_congested[['school_id_origin', 'school_id_destination', 'count_non_beneficiary']].copy()
flow_remaining = flow_remaining.merge(
    lp_redirections_by_flow.reset_index(),
    on=['school_id_origin', 'school_id_destination'],
    how='left'
)
flow_remaining['count_redirected_lp'] = flow_remaining['count_redirected_lp'].fillna(0)
flow_remaining['remaining_non_benef'] = (flow_remaining['count_non_beneficiary'] - flow_remaining['count_redirected_lp']).clip(lower=0)
flow_remaining['remaining_aisle'] = flow_remaining['school_id_destination'].map(remaining_aisle_lp).fillna(0)
flow_remaining['max_additional'] = flow_remaining[['remaining_non_benef', 'remaining_aisle']].min(axis=1)

flows_with_demand = flow_remaining[flow_remaining['max_additional'] > 0].copy()
print(f"Flows with remaining redirectable students: {len(flows_with_demand):,}")
print(f"Total remaining redirectable students: {flows_with_demand['max_additional'].sum():,.0f}")

Flows with remaining redirectable students: 17,020
Total remaining redirectable students: 249,791


In [36]:
# Step 5: Get constrained options and calculate priority scores
# Include size_bonus for consistent priority scoring with main optimization
constrained_options = redirection_options[redirection_options['private_esc_id'].isin(constrained_schools)].copy()
constrained_options = constrained_options[['school_id_origin', 'school_id_destination', 'private_esc_id',
                                            'distance_to_private_km', 'norm_tuition', 'tuition_jhs', 'size_bonus']].copy()

# Normalize distance within constrained options
dist_min = constrained_options['distance_to_private_km'].min()
dist_max = constrained_options['distance_to_private_km'].max()
constrained_options['norm_distance'] = (constrained_options['distance_to_private_km'] - dist_min) / (dist_max - dist_min + 1e-6)

# Priority score: lower = better (nearer + cheaper + LARGER schools prioritized)
# Formula matches Section 3.7: norm_distance + TUITION_WEIGHT * norm_tuition - SIZE_WEIGHT * size_bonus
constrained_options['priority_score'] = (
    constrained_options['norm_distance'] +
    TUITION_WEIGHT * constrained_options['norm_tuition'] -
    SIZE_WEIGHT * constrained_options['size_bonus']
)

print(f"Constrained options to process: {len(constrained_options):,}")

Constrained options to process: 284,407


In [37]:
# Step 6: VECTORIZED - Merge flows with constrained options and find best per flow
# Each flow is assigned to exactly ONE best private school option (lowest priority_score)
merged = flows_with_demand[['school_id_origin', 'school_id_destination', 'max_additional']].merge(
    constrained_options[['school_id_origin', 'school_id_destination', 'private_esc_id', 'priority_score', 'tuition_jhs']],
    on=['school_id_origin', 'school_id_destination'],
    how='inner'
)

print(f"Merged flow-options: {len(merged):,}")

if len(merged) > 0:
    # Find best option per flow (minimum priority_score)
    idx_best = merged.groupby(['school_id_origin', 'school_id_destination'])['priority_score'].idxmin()
    assignments_df = merged.loc[idx_best].copy()
    assignments_df.rename(columns={'max_additional': 'unique_blocked_demand'}, inplace=True)
    
    print(f"Flows assigned to constrained schools: {len(assignments_df):,}")
    print(f"Total unique blocked demand: {assignments_df['unique_blocked_demand'].sum():,.0f}")
    
    # Step 7: Aggregate by private school
    school_agg = assignments_df.groupby('private_esc_id').agg({
        'unique_blocked_demand': 'sum',
        'school_id_destination': 'nunique',
        'school_id_origin': 'count',
        'priority_score': 'mean',
        'tuition_jhs': 'first'
    }).reset_index()
    school_agg.columns = ['private_esc_id', 'unique_blocked_demand', 'num_congested_jhs_affected', 
                          'num_assigned_flows', 'avg_priority_score', 'tuition_jhs']
else:
    print("No flows could be assigned to constrained schools")
    school_agg = pd.DataFrame(columns=['private_esc_id', 'unique_blocked_demand', 
                                        'num_congested_jhs_affected', 'num_assigned_flows', 
                                        'avg_priority_score', 'tuition_jhs'])

Merged flow-options: 194,168
Flows assigned to constrained schools: 13,602
Total unique blocked demand: 219,012


In [38]:
# Step 8: Build final blocked demand analysis table
blocked_demand_analysis = capacity_info.merge(school_agg, on='private_esc_id', how='inner')

# Add school info
blocked_demand_analysis = blocked_demand_analysis.merge(
    sch_info_slim[['school_id', 'school_name', 'division', 'old_region']],
    left_on='private_esc_id',
    right_on='school_id',
    how='left'
).drop(columns='school_id')
blocked_demand_analysis.rename(columns={'school_name': 'private_esc_name', 'old_region': 'region'}, inplace=True)

# Add school size data                                                                                    
blocked_demand_analysis = blocked_demand_analysis.merge(                                                  
  private_jhs_enrollment[['school_id', 'private_jhs_enrollment', 'size_bonus']],                        
  left_on='private_esc_id',                                                                             
  right_on='school_id',                                                                                 
  how='left'                                                                                            
).drop(columns='school_id')

# Reorder and sort
blocked_demand_analysis = blocked_demand_analysis[[                                                       
  'private_esc_id', 'private_esc_name', 'division', 'region',                                           
  'total_slots', 'available_slots_before', 'slots_used_lp', 'utilization_rate',                         
  'unique_blocked_demand', 'num_assigned_flows', 'num_congested_jhs_affected',                          
  'tuition_jhs', 'private_jhs_enrollment', 'size_bonus', 'avg_priority_score'                           
]]

blocked_demand_analysis = blocked_demand_analysis.sort_values(
    ['unique_blocked_demand', 'avg_priority_score'], 
    ascending=[False, True]
)

# Summary
print("="*70)
print("BLOCKED DEMAND ANALYSIS SUMMARY")
print("="*70)
print(f"Capacity-constrained schools (>=95% utilized): {len(constrained_schools):,}")
print(f"Schools with blocked demand: {len(blocked_demand_analysis):,}")
print(f"Total unique blocked demand: {blocked_demand_analysis['unique_blocked_demand'].sum():,.0f} students")
print(f"\nTop 10 schools for slot expansion:")
print("-"*70)
display(blocked_demand_analysis[['private_esc_name', 'division', 'slots_used_lp',                         
                                'unique_blocked_demand', 'tuition_jhs', 'private_jhs_enrollment', 'size_bonus']].head(10))

BLOCKED DEMAND ANALYSIS SUMMARY
Capacity-constrained schools (>=95% utilized): 931
Schools with blocked demand: 574
Total unique blocked demand: 219,012 students

Top 10 schools for slot expansion:
----------------------------------------------------------------------


,private_esc_name,division,slots_used_lp,unique_blocked_demand,tuition_jhs,private_jhs_enrollment,size_bonus
547,"St. Anthony Learning Institute of Quezon City,...",Quezon City,2.0,3181.0,15375.00,130.0,0.554537
176,Sumulong Memorial High School,Antipolo City,125.0,3001.0,26361.88,1704.0,0.997954
373,"Holy Rosary College Foundation, Inc.",Caloocan City,88.0,2906.0,22024.91,320.0,0.814680
466,Holy Child Catholic School,Manila,111.0,2582.0,45050.00,930.0,0.975979
296,Valley High Academy,Rizal,19.0,2316.0,27750.00,166.0,0.627135
131,Holy Redeemer School,Dasmarinas City,102.0,2198.0,24775.00,261.0,0.758808
467,"Manila Cathedral College, Inc.",Manila,49.0,2148.0,43414.25,1210.0,0.991459
217,St. Gabriel Archangel Academy (Main),Binan City,1.0,1953.0,36300.00,56.0,0.329537
384,Sta. Clara Parish School,Pasay City,49.0,1892.0,46771.22,374.0,0.851157
242,Golden Faith Academy Inc.,Rizal,50.0,1876.0,42300.00,357.0,0.840125


---
# 7.0. Export Payload

All key dataframes and metadata are saved to `output/analysis_payload/` for consumption by **notebook 2.5** (stakeholder report generation).

This separation of concerns — analysis in 2.4, reporting in 2.5 — means we can regenerate reports without re-running the optimization.

**Exported files (10 total):**

| File | Rows | Purpose |
|------|------|---------|
| `congestion.parquet` | ~1,098 | Public JHS congestion metrics |
| `redirections_greedy.parquet` | ~5,070 | Greedy redirection log |
| `redirections_lp.parquet` | ~6,162 | LP redirection log |
| `redirection_options.parquet` | ~284,670 | All feasible options with priority scores |
| `esc_available.parquet` | ~1,058 | ESC slot availability |
| `flow_to_congested.parquet` | ~22,879 | Student flows to congested destinations |
| `school_info.parquet` | ~61,442 | School metadata |
| `student_flow.parquet` | ~57,261 | Complete flow data (NCR/IV-A) |
| `blocked_demand_analysis.parquet` | ~574 | Slot expansion priorities |
| `metadata.json` | — | Parameters and summary statistics |

In [ ]:
# # Create output directory for scenario1 payload
# PAYLOAD_DIR = 'output/analysis_payload'
# os.makedirs(PAYLOAD_DIR, exist_ok=True)

# # 1. Congestion table (public JHS with enrollment, seats, aisle learners)
# df_cong.to_parquet(f'{PAYLOAD_DIR}/congestion.parquet', index=False)
# print(f"✓ Exported congestion.parquet ({df_cong.shape[0]:,} rows)")

# # 2. Greedy redirection log
# redirections.to_parquet(f'{PAYLOAD_DIR}/redirections_greedy.parquet', index=False)
# print(f"✓ Exported redirections_greedy.parquet ({redirections.shape[0]:,} rows)")

# # 3. LP redirection log
# redirections_lp.to_parquet(f'{PAYLOAD_DIR}/redirections_lp.parquet', index=False)
# print(f"✓ Exported redirections_lp.parquet ({redirections_lp.shape[0]:,} rows)")

# # 4. All redirection options (with priority_score, size_bonus, etc.)
# redirection_options.to_parquet(f'{PAYLOAD_DIR}/redirection_options.parquet', index=False)
# print(f"✓ Exported redirection_options.parquet ({redirection_options.shape[0]:,} rows)")

# # 5. ESC slots and availability
# esc_available.to_parquet(f'{PAYLOAD_DIR}/esc_available.parquet', index=False)
# print(f"✓ Exported esc_available.parquet ({esc_available.shape[0]:,} rows)")

# # 6. Filtered student flows (to congested schools)
# flow_to_congested.to_parquet(f'{PAYLOAD_DIR}/flow_to_congested.parquet', index=False)
# print(f"✓ Exported flow_to_congested.parquet ({flow_to_congested.shape[0]:,} rows)")

# # 7. School info (for names/divisions in reports)
# sch_info_slim.to_parquet(f'{PAYLOAD_DIR}/school_info.parquet', index=False)
# print(f"✓ Exported school_info.parquet ({sch_info_slim.shape[0]:,} rows)")

# # 8. Student flow (full, for baseline calculations)
# flow.to_parquet(f'{PAYLOAD_DIR}/student_flow.parquet', index=False)
# print(f"✓ Exported student_flow.parquet ({flow.shape[0]:,} rows)")

# # 9. Blocked demand analysis (slot expansion priorities)
# blocked_demand_analysis.to_parquet(f'{PAYLOAD_DIR}/blocked_demand_analysis.parquet', index=False)
# print(f"✓ Exported blocked_demand_analysis.parquet ({blocked_demand_analysis.shape[0]:,} rows)")

# # 10. Metadata (parameters and summary stats)
# metadata = {
#     'max_distance_km': MAX_DISTANCE_KM,
#     'tuition_weight': TUITION_WEIGHT,
#     'size_weight': SIZE_WEIGHT,
#     'cost_penalty': COST_PENALTY,
#     'distance_penalty': DISTANCE_PENALTY,
#     'size_weight_lp': SIZE_WEIGHT_LP,
#     'baseline_aisle_learners': int(baseline_aisle),
#     'greedy_aisle_learners': int(greedy_aisle),
#     'lp_aisle_learners': int(lp_aisle),
#     'greedy_students_redirected': int(redirections['count_redirected'].sum()),
#     'lp_students_redirected': int(redirections_lp['count_redirected_lp'].sum()),
#     'greedy_schools_used': int(redirections['private_esc_id'].nunique()),
#     'lp_schools_used': int(redirections_lp['private_esc_id'].nunique()),
#     'constrained_schools': len(constrained_schools),
#     'total_blocked_demand': int(blocked_demand_analysis['unique_blocked_demand'].sum()),
# }
# with open(f'{PAYLOAD_DIR}/metadata.json', 'w') as f:
#     json.dump(metadata, f, indent=2)
# print(f"✓ Exported metadata.json")

# print(f"\n{'='*50}")
# print(f"Payload exported to: {PAYLOAD_DIR}/")
# print(f"{'='*50}")

---
## Summary of Key Results

| Metric | Value |
|--------|-------|
| Public JHS analyzed | ~1,098 |
| Congested schools | ~715 (65%) |
| **Baseline aisle learners** | **~577,700** |
| ESC slots available | ~27,100 |
| Students redirected (Greedy) | ~25,800 |
| Students redirected (LP) | ~25,900 |
| **Aisle learner reduction** | **~4.5%** |
| Avg redirect distance (LP) | ~1.0 km |
| Capacity-constrained ESC schools | ~931 |

**The bottleneck is clear:** with only ~27K available ESC slots, we can redirect at most ~26K students — a 4.5% dent in ~578K aisle learners. To achieve a 10%+ reduction, we'd need roughly 55K+ additional ESC slots.

## Open Questions for Team Discussion

1. **Distance threshold:** Is 5 km appropriate for both NCR (dense, good transit) and CALABARZON (suburban, car-dependent)?
2. **Priority weights:** Should we calibrate distance/tuition/size weights from actual family behavior data?
3. **Shift systems:** How does the morning/afternoon shift system affect our aisle learner counts?
4. **Slot expansion:** The blocked demand analysis identifies top schools for new slots — how do we validate these recommendations with DepEd regional offices?
5. **Multi-year dynamics:** This is a single-year snapshot. How do enrollment trends affect the recommendation?